In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_16_smart_and_force_bias_mc.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Atomic Scale Simulations · Lesson 16: Smarter Monte Carlo — Menus, Preferential, Heat-Bath, Force-Bias & Smart MC

_Session 2a4425f6 · Estimated time: ~115 minutes · 38 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Distinguish between detailed balance and global stationary balance when combining a menu of moves via random selection versus deterministic sequential cycling.
- Formulate preferential sampling for inhomogeneous/layered systems and derive the exact asymmetric proposal ratio involving H(R) and H(R').
- Derive the Heat-Bath (Gibbs) local conditional transition rule T(s -> s') = pi(s')/C(s) and explain when its acceptance probability A = min(1, C(s)/C(s')) equals 100%.
- Construct Force-Bias Monte Carlo inside a trial cube using local forces F(r) = -grad V(r) and inverse-CDF exponential mapping.
- Derive the Smart Monte Carlo displaced-Gaussian proposal and acceptance exponent, prove that A = 1 identically for any linear potential, and connect Smart MC to Brownian Dynamics and Diffusion Quantum Monte Carlo.


In [2]:
ws.render_section('sec_menus_of_moves')

## 1. Menus of Moves: Random Selection vs. Deterministic Sequential Sweeps

In Lesson 15 we established the "classic" Metropolis method using a single type of move with a uniform proposal $T(s \to s') = \text{constant}$. While convergence to $\pi(s) = e^{-\beta E(s)}/Z$ is guaranteed for any ergodic walk, **the rate of convergence is not!** Today we explore five powerful strategies to redesign $T(s \to s')$ for faster convergence:
1. **Menus of Moves**
2. **Preferential Sampling**
3. **Heat-Bath Monte Carlo**
4. **Force-Bias Monte Carlo**
5. **Smart Monte Carlo & Hybrid Brownian Dynamics**

### Global Balance vs. Detailed Balance Revisited
Recall that for $\pi(s)$ to be the unique stationary distribution of an ergodic Markov chain $P(s \to s')$, it is **not** strictly required that $P$ satisfy pairwise detailed balance $\pi(s)P(s \to s') = \pi(s')P(s' \to s)$. Rather, **global balance (stationarity)** suffices:
$$\pi(s) = \sum_{s'} \pi(s')\, P(s' \to s) \qquad \Longleftrightarrow \qquad \pi = \pi P$$

### Combining a Menu of Moves $P_A$ and $P_B$
Suppose we have two (or more) distinct move types with transition matrices $P_A$ and $P_B$—for instance, moving particle $1$ ($P_A$) and moving particle $2$ ($P_B$), or a particle displacement ($P_A$) and a box volume change ($P_B$) in an $NPT$ simulation—each of which **individually satisfies detailed balance** for the same target distribution $\pi$:
$$\pi P_A = \pi \qquad \text{and} \qquad \pi P_B = \pi$$
We can combine $(A, B)$ in two ways:
1. **Random Selection (Convex Combination):** At each step, choose move $A$ with probability $\alpha_A$ and move $B$ with probability $\alpha_B = 1 - \alpha_A$. The single-step transition matrix is:
 $$P_{\text{rand}} = \alpha_A P_A + \alpha_B P_B$$
 Because detailed balance is linear in $P$, $P_{\text{rand}}$ **satisfies both detailed balance and global balance**:
 $$\pi(s) P_{\text{rand}}(s \to s') = \alpha_A \pi(s)P_A(s \to s') + \alpha_B \pi(s)P_B(s \to s') = \pi(s') P_{\text{rand}}(s' \to s)$$
2. **Deterministic Sequential Cycling $(A, B, A, B, A, B, \dots)$:** Apply move $A$ followed deterministically by move $B$. The composite two-step transition matrix for one full cycle is the matrix product:
 $$P_{\text{seq}} = P_A P_B$$
 - **Does $P_{\text{seq}} = P_A P_B$ satisfy detailed balance?** **No!** Let $D_\pi = \operatorname{diag}(\pi)$. Detailed balance in matrix form is $D_\pi P = P^T D_\pi$. Taking the transpose of $D_\pi P_A P_B$ gives:
 $$(D_\pi P_A P_B)^T = P_B^T (D_\pi P_A)^T = P_B^T D_\pi P_A = D_\pi P_B P_A \ne D_\pi P_A P_B$$
 whenever $P_A$ and $P_B$ do not commute ($P_A P_B \ne P_B P_A$). Physically, reversing the movie of an $(A, B)$ cycle produces a $(B, A)$ cycle!
 - **Does $P_{\text{seq}} = P_A P_B$ preserve the exact stationary state $\pi$?** **Yes!** Applying $\pi$ from the left associative-style gives:
 $$\pi P_{\text{seq}} = (\pi P_A) P_B = \pi P_B = \pi$$
 - **Theorem:** *If each menu item individually satisfies detailed balance (andhence preserves $\pi$), then $\pi$ is the unique stationary state of the deterministic cycle $P_1 P_2 \cdots P_M$ (provided the combined chain is ergodic).*
 - **Practical Example:** In classic $N$-particle Metropolis MC, instead of calling a random number generator to pick a random particle index $i \in \{1, \dots, N\}$ at every step, you can simply **loop sequentially through $i = 1, 2, \dots, N$**. After one sweep (pass), every particle has been attempted once, saving $N$ random-integer draws per pass while converging to the exact Boltzmann distribution!


In [3]:
ws.render_section('q16_01_global_vs_detailed_balance_menu')

**Q1 · easy** · 💡

As emphasized and 6, why is a deterministic cycle of moves $(A, B, A, B, \dots)$ valid in Markov Chain Monte Carlo even though the composite transition matrix $P = P_A P_B$ violates detailed balance?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\pi (P_A P_B)$ when $\pi P_A = \pi$ and $\pi P_B = \pi$.

</div>
</details>

- [ ] **(A)** Because global stationary balance $\pi = \pi P$ is what guarantees the equilibrium distribution, and $\pi(P_A P_B) = (\pi P_A)P_B = \pi P_B = \pi$ holds automatically whenever each individual move preserves $\pi$.
- [ ] **(B)** Because any two stochastic matrices $P_A$ and $P_B$ always commute ($P_A P_B = P_B P_A$).
- [ ] **(C)** Because detailed balance is only violated during warmup and becomes exact once $n > 100$.
- [ ] **(D)** Because deterministic cycling sets all rejection probabilities to zero.

In [4]:
ws.render_section('q16_02_why_product_violates_detailed_balance')

**Q2 · medium** · 💡

Suppose $P_A$ and $P_B$ each satisfy detailed balance with respect to $\pi$ ($D_\pi P_A = P_A^T D_\pi$ and $D_\pi P_B = P_B^T D_\pi$, where $D_\pi = \operatorname{diag}(\pi)$). Under what algebraic condition does the sequential product $P_{\text{seq}} = P_A P_B$ also satisfy detailed balance?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $(D_\pi P_A P_B)^T = D_\pi P_B P_A$ with $D_\pi P_A P_B$.

</div>
</details>

- [ ] **(A)** Only if $P_A$ and $P_B$ commute ($P_A P_B = P_B P_A$), or if the sequence is palindromically symmetrized such as $P_A P_B P_A$.
- [ ] **(B)** Always, because the product of two symmetric matrices is always symmetric.
- [ ] **(C)** Never, even if $P_A = P_B$.
- [ ] **(D)** Only when $\pi(s)$ is the uniform distribution.

In [5]:
ws.render_section('q16_03_random_menu_convex_combination')

**Q3 · easy** · 💡

In an isothermal-isobaric ($NPT$) Monte Carlo simulation, at each step you randomly select a particle translation move ($P_{\text{trans}}$) with fixed probability $\alpha = 0.98$ or a volume-scaling move ($P_{\text{vol}}$) with probability $1 - \alpha = 0.02$. If both $P_{\text{trans}}$ and $P_{\text{vol}}$ satisfy detailed balance for the $NPT$ ensemble, what is true of the combined single-step transition matrix $P = 0.98 P_{\text{trans}} + 0.02 P_{\text{vol}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check whether a linear combination of equations satisfying $\pi(s)P_k(s\to s') = \pi(s')P_k(s'\to s)$ still satisfies detailed balance.

</div>
</details>

- [ ] **(A)** $P$ satisfies both detailed balance and global balance for the $NPT$ distribution.
- [ ] **(B)** $P$ satisfies global balance but violates detailed balance unless $\alpha = 0.50$.
- [ ] **(C)** $P$ is not a stochastic matrix because its rows sum to $1.98$.
- [ ] **(D)** $P$ biases the equilibrium volume by a factor of $0.02 / 0.98$.

In [6]:
ws.render_section('q16_04_sequential_particle_sweep')

**Q4 · easy** · 💡

In a canonical $NVT$ simulation of $N$ atoms, instead of choosing a particle index $i \in \{1, \dots, N\}$ at random on each trial step, a programmer loops deterministically through $i = 1, 2, \dots, N$ in each sweep. According to the Menu of Moves theorem, is this procedure correct?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each single-particle transition matrix $P_i$ individually satisfies detailed balance for the canonical Boltzmann distribution $\pi(\mathbf{R})$.

</div>
</details>

- [ ] **(A)** Yes: because each single-particle transition operator $P_i$ individually satisfies detailed balance for $\pi(\mathbf{R})$, the composite sweep $P_{\text{sweep}} = P_1 P_2 \cdots P_N$ has $\pi(\mathbf{R})$ as its exact stationary distribution and guarantees every particle is attempted once per pass.
- [ ] **(B)** No: deterministic looping over particle indices introduces an $\mathcal{O}(1/N)$ bias into the radial distribution function $g(r)$.
- [ ] **(C)** Yes, but only if the particles are sorted by their $x$-coordinate before every pass.
- [ ] **(D)** No: it violates the Markov property unless all $N$ moves are accepted simultaneously.

In [7]:
ws.render_section('q16_05_non_commuting_2x2_product_calc')

**Q5 · medium** · 💡

Suppose two move operators $P_A$ and $P_B$ on a 3-state system with uniform target distribution $\pi = (1/3, 1/3, 1/3)$ are given by the symmetric permutation-mixing matrices:
$$P_A = \begin{pmatrix} 0.5 & 0.5 & 0 \\ 0.5 & 0.5 & 0 \\ 0 & 0 & 1 \end{pmatrix}, \qquad P_B = \begin{pmatrix} 1 & 0 & 0 \\ 0 & 0.5 & 0.5 \\ 0 & 0.5 & 0.5 \end{pmatrix}$$
Consider the composite sequential transition matrix $P_{\text{seq}} = P_A P_B$. What is the transition probability $P_{\text{seq}}(1 \to 3)$ from state $1$ to state $3$ in a single $(A, B)$ cycle?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Perform matrix multiplication to find the $(1, 3)$ element of the composite product matrix $P_A P_B$.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q16_06_state_dependent_menu_warning')

**Q6 · hard** · 💡

When randomly selecting move type $k$ from a menu with probability $\alpha_k$, what happens if you make the selection probabilities $\alpha_k(s)$ depend on the current microstate $s$ without including $\alpha_k(s')/\alpha_k(s)$ in the acceptance test?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check whether $\pi(s) \alpha_k(s) P_k(s \to s')$ equals $\pi(s') \alpha_k(s') P_k(s' \to s)$ when $\alpha_k(s) \ne \alpha_k(s')$.

</div>
</details>

- [ ] **(A)** Detailed balance and global balance are broken unless the state-dependent selection ratio $\alpha_k(s')/\alpha_k(s)$ is included inside the Metropolis–Hastings acceptance ratio (as in Preferential Sampling).
- [ ] **(B)** Nothing changes; $\alpha_k(s)$ cancels out automatically even when $\alpha_k(s) \ne \alpha_k(s')$.
- [ ] **(C)** The acceptance probability becomes $1.0$ for all moves.
- [ ] **(D)** The chain becomes periodic with eigenvalue $\varepsilon = -1$.

In [9]:
ws.render_section('q16_07_Sweep_coverage_probability')

**Q7 · medium** · 💡

In an $N$-particle fluid ($N \gg 1$), one pass consists of $N$ single-particle trial moves. If a particle index is chosen uniformly at random with replacement on each of the $N$ attempts, what fraction of the $N$ particles are **not** attempted at all during a single pass in the thermodynamic limit $N \to \infty$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate the probability that a specific particle is missed across $N$ independent random selections with replacement as $N \to \infty$.

</div>
</details>

_Numeric answer required._

In [10]:
ws.render_section('sec_preferential_sampling')

### Focusing CPU Time on Interesting Degrees of Freedom
Sometimes it is **not** the overall bulk acceptance ratio that we want to optimize! Consider a layered slab, a catalytic crystal surface, a solid-liquid interface, or a biomolecule in a large water box:
- The vast majority of atoms sit deep in the uninteresting bulk substrate or far-away solvent, while the physical properties we care about (adsorption, surface reconstruction, step-edge diffusion) depend on atoms near the **top surface** ($z \approx z_{\text{top}}$).
- If we pick particles uniformly at random ($T_i = 1/N$), $95\%$ of our CPU time is wasted moving bulk atoms!

### Preferential Selection Probability $h(z_i) / H(\mathbf{R})$
To focus MC moves on the region of interest while preserving exact canonical equilibrium everywhere, we assign each particle $i$ at height $z_i$ a strictly positive weight function $h(z_i) > 0$ (e.g., large near the surface and small in the bulk) and select particle $i$ with normalized probability:
$$T_i(\mathbf{R}) = \frac{h(z_i)}{H(\mathbf{R})}, \qquad \text{where} \quad H(\mathbf{R}) = \sum_{j=1}^N h(z_j)$$
Once particle $i$ is selected, we displace it uniformly inside a symmetric cube of volume $\Delta^3$ from $\mathbf{r}_i \to \mathbf{r}_i'$ (changing its height from $z_i \to z_i'$).

### Why the Normalization Sum $H(\mathbf{R}') \ne H(\mathbf{R})$ Changes!
Crucially, because particle $i$ moved from $z_i$ to $z_i'$, its weight changed from $h(z_i)$ to $h(z_i')$. Therefore, the total normalization sum in the trial configuration $\mathbf{R}'$ is slightly different from $H(\mathbf{R})$:
$$H(\mathbf{R}') = \sum_{j \ne i} h(z_j) + h(z_i') = H(\mathbf{R}) - h(z_i) + h(z_i')$$
- **$\mathcal{O}(1)$ Update Trick:** Notice that we do **not** need $\mathcal{O}(N)$ operations to recompute $H(\mathbf{R}')$ at every step; we simply subtract $h(z_i)$ and add $h(z_i')$!
- **Asymmetric Proposal Ratio:**
 - Forward proposal density ($\mathbf{R} \to \mathbf{R}'$): $T(\mathbf{R} \to \mathbf{R}') = \frac{h(z_i)}{H(\mathbf{R})} \frac{1}{\Delta^3}$
 - Reverse proposal density ($\mathbf{R}' \to \mathbf{R}$): $T(\mathbf{R}' \to \mathbf{R}) = \frac{h(z_i')}{H(\mathbf{R}')} \frac{1}{\Delta^3}$
 Substituting into the generalized Metropolis–Hastings formula $A(\mathbf{R} \to \mathbf{R}') = \min\!\left\{1, \frac{T(\mathbf{R}' \to \mathbf{R})\pi(\mathbf{R}')}{T(\mathbf{R} \to \mathbf{R}')\pi(\mathbf{R})}\right\}$ gives:
 $$A(\mathbf{R} \to \mathbf{R}') = \min\!\left\{1,\; \frac{h(z_i')}{h(z_i)} \frac{H(\mathbf{R})}{H(\mathbf{R}')}\, e^{-\beta[V(\mathbf{R}') - V(\mathbf{R})]}\right\}$$
 *(Note: writes the prefactor schematically as $\frac{h(z_i)}{H}\frac{H'}{h(z_i')}$ by swapping forward/reverse labels, but in code `A = exp(-dV) * (T_old / T_new)`, $T_{\text{old}} = T(\mathbf{R}' \to \mathbf{R}) = h(z_i')/H'$ is in the numerator and $T_{\text{new}} = T(\mathbf{R} \to \mathbf{R}') = h(z_i)/H$ is in the denominator!)*
- **Physical Consequence:** When a frequently sampled surface atom ($h(z_i)$ large) proposes a move into the rarely sampled bulk ($h(z_i')$ small), the ratio $T(\mathbf{R}' \to \mathbf{R}) / T(\mathbf{R} \to \mathbf{R}') \approx h(z_i')/h(z_i) < 1$ **lowers the acceptance probability** of surface-to-bulk moves so that particles do not artificially drain out of the surface simply because they are poked more often!
- **Ergodicity Requirement:** We must keep $h(z) > 0$ everywhere in the system (never strictly zero in the bulk) if bulk coordinates are meant to equilibrate with the surface.


In [11]:
ws.render_section('concept_preferential_surface')

### Interactive Explorer: Preferential Surface Sampling & The $H/H'$ Jacobian Ratio

Adjust the surface decay length $z_0$, initial particle height $z_i$, and vertical trial displacement $\Delta z$ to observe how preferential weighting $h(z)$ focuses trial moves near the top surface ($z_{\mathrm{top}} = 10$) and how the reverse-to-forward proposal ratio $\frac{h(z_i')/H'}{h(z_i)/H}$ preserves uniform equilibrium density.

**Key Takeaway:** Because surface atoms are selected much more frequently than bulk atoms ($h(z_{\mathrm{surf}}) \gg h(z_{\mathrm{bulk}})$), a naive symmetric acceptance test would pump atoms out of the surface into the bulk. Multiplying by $T(\mathbf{R}'\to\mathbf{R})/T(\mathbf{R}\to\mathbf{R}') = [h(z_i')/H']/[h(z_i)/H]$ rejects excess surface-to-bulk hops and preserves the exact Boltzmann distribution.

In [12]:
ws.render_section('q16_08_preferential_sampling_goal')

**Q8 · easy** · 💡

What is the primary physical motivation for using **Preferential Sampling** in a layered or interfacial system?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how CPU time is distributed between active surface degrees of freedom and passive bulk atoms.

</div>
</details>

- [ ] **(A)** To focus CPU time on sampling the most physically interesting degrees of freedom (such as a surface or interface), increasing the statistical efficiency of surface observables even though overall move acceptance is slightly lowered.
- [ ] **(B)** To change the equilibrium thermodynamic ensemble from canonical ($NVT$) to microcanonical ($NVE$).
- [ ] **(C)** To make the equilibrium density near the surface higher than predicted by the Boltzmann distribution.
- [ ] **(D)** To eliminate the need to calculate potential energy differences $\Delta V$.

In [13]:
ws.render_section('q16_09_preferential_H_prime_update')

**Q9 · easy** · 💡

In a preferential sampling simulation of $N = 500$ atoms, the current normalization sum of selection weights is $H(\mathbf{R}) = \sum_{j=1}^N h(z_j) = 40.0$. Atom $i$ is selected at height $z_i$ where $h(z_i) = 0.80$, and is proposed to move to height $z_i'$ where $h(z_i') = 0.30$. What is the new normalization sum $H(\mathbf{R}')$ in the trial configuration?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Account for the replacement of the moved particle's old weight with its new weight in the cumulative sum.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q16_10_preferential_acceptance_numeric')

**Q10 · medium** · 💡

Using the numbers from the previous question ($H(\mathbf{R}) = 40.0$, $h(z_i) = 0.80$, $H(\mathbf{R}') = 39.5$, $h(z_i') = 0.30$), suppose the trial move from $z_i$ to $z_i'$ has zero potential energy change ($\Delta V = V(\mathbf{R}') - V(\mathbf{R}) = 0$). What is the exact Metropolis–Hastings acceptance probability
$$A(\mathbf{R} \to \mathbf{R}') = \min\!\left\{1,\; \frac{T(\mathbf{R}' \to \mathbf{R})}{T(\mathbf{R} \to \mathbf{R}')} e^{-\beta \Delta V}\right\} = \min\!\left\{1,\; \frac{h(z_i')/H(\mathbf{R}')}{h(z_i)/H(\mathbf{R})}\right\}$$
for this surface-to-bulk move?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the ratio of reverse to forward selection probabilities using the individual weights and their respective partition sums.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q16_11_why_preferential_lowers_acceptance')

**Q11 · medium** · 💡

Consider an ideal gas ($\Delta V = 0$ everywhere) in a slab where surface atoms ($z > 8$) are given selection weight $h_{\text{surf}} = 10$ and bulk atoms ($z \le 8$) are given selection weight $h_{\text{bulk}} = 1$. Why must the Metropolis acceptance test reject $\sim 90\%$ of moves that carry an atom from the surface into the bulk, while accepting $100\%$ of moves that carry an atom from the bulk into the surface?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In an ideal gas at equilibrium, the average particle density must remain uniform between bulk and surface.

</div>
</details>

- [ ] **(A)** Because surface atoms are selected $10\times$ more often than bulk atoms; rejecting $90\%$ of surface-to-bulk proposals balances the $10\times$ higher proposal attempt frequency so the net flux between surface and bulk is equal in both directions.
- [ ] **(B)** Because ideal gas atoms experience a strong attractive force toward the top surface.
- [ ] **(C)** Because bulk atoms have a higher temperature than surface atoms.
- [ ] **(D)** Because the trial cube volume $\Delta^3$ is $10\times$ smaller in the bulk.

In [16]:
ws.render_section('q16_12_preferential_ergodicity_condition')

**Q12 · easy** · 💡

If you want all $N$ particles in a slab (both surface and bulk) to equilibrate into the true canonical ensemble under preferential sampling $T_i(\mathbf{R}) = h(z_i)/H(\mathbf{R})$, what condition must the weight function $h(z)$ satisfy?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the irreducibility condition for ergodicity when particles can exchange between bulk and surface.

</div>
</details>

- [ ] **(A)** $h(z) > 0$ strictly everywhere in the accessible volume so that every particle has a nonzero probability of being selected and reverse moves from any $z'$ to $z$ remain possible.
- [ ] **(B)** $h(z)$ must be negative in the bulk and positive at the surface.
- [ ] **(C)** $h(z)$ must equal a Dirac delta function $\delta(z - z_{\text{top}})$.
- [ ] **(D)** $h(z)$ must be an oscillatory sine wave $\sin(kz)$.

In [17]:
ws.render_section('q16_13_horizontal_move_in_preferential_slab')

**Q13 · easy** · 💡

In a preferential sampling simulation where the selection weight $h(z_i)$ depends **only** on the vertical height $z_i$, suppose particle $i$ makes a purely horizontal trial displacement $(\Delta x, \Delta y, \Delta z = 0)$ so that $z_i' = z_i$. What is the value of the proposal ratio $\frac{T(\mathbf{R}' \to \mathbf{R})}{T(\mathbf{R} \to \mathbf{R}')} = \frac{h(z_i')/H(\mathbf{R}')}{h(z_i)/H(\mathbf{R})}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $h(z_i')$ with $h(z_i)$ and $H(\mathbf{R}')$ with $H(\mathbf{R})$ when $z_i' = z_i$.

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q16_14_why_H_cannot_be_approximated_as_constant')

**Q14 · medium** · 💡

Why does the preferential acceptance formula include the ratio $H(\mathbf{R})/H(\mathbf{R}')$ in addition to $h(z_i')/h(z_i)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that the selection probabilities across the $N$ particles must sum to $1$ in both configuration $\mathbf{R}$ and configuration $\mathbf{R}'$.

</div>
</details>

- [ ] **(A)** Because normalizing the discrete selection distribution over the $N$ particles requires dividing by $H(\mathbf{R}) = \sum_{j=1}^N h(z_j)$, which changes to $H(\mathbf{R}') = H(\mathbf{R}) - h(z_i) + h(z_i')$ when particle $i$ changes its height.
- [ ] **(B)** Because $H(\mathbf{R})$ is the total Hamiltonian energy of the system.
- [ ] **(C)** Because $H(\mathbf{R})/H(\mathbf{R}')$ is always equal to $0$.
- [ ] **(D)** Because the number of particles $N$ changes during a canonical translation move.

In [19]:
ws.render_section('sec_heat_bath_mc')

### The Dream of $100\%$ Acceptance: $T(s \to s') \propto \pi(s')$
Look again at the generalized Metropolis–Hastings acceptance formula:
$$A(s \to s') = \min\!\left\{1,\; \frac{T(s' \to s)\,\pi(s')}{T(s \to s')\,\pi(s)}\right\}$$
Suppose we could choose the proposal distribution $T(s \to s')$ to be directly proportional to the target equilibrium probability of the destination state:
$$T(s \to s') = \frac{\pi(s')}{C}$$
where $C$ is a constant independent of $s$. Then the ratio inside $\min\{\dots\}$ would be:
$$\frac{[\pi(s)/C]\,\pi(s')}{[\pi(s')/C]\,\pi(s)} = 1 \quad \Longrightarrow \quad A(s \to s') = 1 \quad (100\%\text{ acceptance!})$$
If $\mathcal{N}(s)$ were the entire phase space, $C = \sum_{s''} \pi(s'') = 1$ would correspond to **direct independent sampling** from $\pi(s')$, which we already know is intractable for an $N$-body system.

### Heat-Bath Method: Exact Sampling in a Limited Neighborhood $\mathcal{N}(s)$
While we cannot sample all $3N$ coordinates from $\pi$ at once, we **can** often sample a small **local neighborhood** $\mathcal{N}(s)$ of states (such as flipping a single lattice spin $\sigma_i \in \{-1, +1\}$, choosing among a few discrete crystal site orientations, or redrawing a single coordinate) so that the local degree of freedom is placed immediately into **"local thermal equilibrium"** with its fixed neighbors:
$$T(s \to s') = \frac{\pi(s')}{C(s)} \quad \text{for } s' \in \mathcal{N}(s), \qquad \text{with} \quad C(s) = \sum_{s'' \in \mathcal{N}(s)} \pi(s'')$$
Substituting $T(s \to s') = \pi(s')/C(s)$ and $T(s' \to s) = \pi(s)/C(s')$ (assuming $s \in \mathcal{N}(s')$) into the Metropolis–Hastings acceptance formula cancels $\pi(s)$ and $\pi(s')$, leaving **only the ratio of local normalizations**:
$$A(s \to s') = \min\!\left\{1,\; \frac{C(s)}{C(s')}\right\}$$

### Two Key Operational Rules for Heat-Bath MC
1. **Feasibility condition:** Heat-Bath MC can be used **only if it is possible to quickly compute the local normalization sum $C(s)$** (and $C(s')$) over the neighborhood $\mathcal{N}(s)$.
2. **Rejection-Free ($A = 1$) Condition:** **If $C(s)$ is independent of $s$ ($C(s) = C(s')$), then the acceptance ratio is identically $A(s \to s') = 1$!**
 - *Example (Ising / Lattice Models — Gibbs Sampler):* If move $s \to s'$ consists of updating a single spin $\sigma_i \in \{-1, +1\}$ while keeping all other spins $\sigma_{j \ne i}$ fixed, the neighborhood $\mathcal{N}(s) = \{(\sigma_i = +1, \sigma_{-i}),\; (\sigma_i = -1, \sigma_{-i})\}$ is identical whether we start at $\sigma_i = +1$ or $\sigma_i = -1$! Thus:
 $$C(s) = \pi(\sigma_i = +1) + \pi(\sigma_i = -1) = C(s') \quad \Longrightarrow \quad A(s \to s') = 1$$
 In a local field $h_{\text{loc}} = J\sum_{j \in nn(i)} \sigma_j + H_{\text{ext}}$, Heat-Bath sets $\sigma_i' = +1$ with probability $\frac{e^{+\beta h_{\text{loc}}}}{e^{+\beta h_{\text{loc}}} + e^{-\beta h_{\text{loc}}}} = \frac{1}{1 + e^{-2\beta h_{\text{loc}}}}$ and $\sigma_i' = -1$ otherwise, with no rejection step needed!


In [20]:
ws.render_section('concept_heat_bath')

### Interactive Explorer: Heat-Bath Local Equilibrium vs. Metropolis Spin Flip

Vary the dimensionless local molecular field $\beta h_{\mathrm{loc}}$ and current spin state $\sigma_i \in \{-1, +1\}$ to compare how Heat-Bath directly samples the local conditional equilibrium distribution $T(s\to s') = \pi(s')/C(s)$ with $A=1$, whereas Metropolis proposes a deterministic flip $\sigma_i \to -\sigma_i$ and accepts with $\min(1, e^{-\beta\Delta E})$.

**Key Takeaway:** Heat-Bath places the selected degree of freedom directly into local thermal equilibrium with its neighbors in a single step ($A = 1$ when $C(s) = C(s')$). Notice that for a two-state spin, Metropolis flips state $-\sigma_i$ more aggressively ($P_M = \min(1, e^{-\beta\Delta E}) \ge P_{HB} = (1 + e^{\beta\Delta E})^{-1}$ by Peskun's theorem), whereas Heat-Bath shines for multi-state Potts/lattice models or continuous conditional Gaussians where random blind proposals have tiny acceptance!

In [21]:
ws.render_section('q16_15_heat_bath_proposal_definition')

**Q15 · easy** · 💡

In the **Heat-Bath** Monte Carlo method, how is the trial transition probability $T(s \to s')$ defined for states $s'$ inside a local neighborhood $\mathcal{N}(s)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Normalize the target equilibrium weight $\pi(s')$ over all states $s''$ in the neighborhood $\mathcal{N}(s)$.

</div>
</details>

- [ ] **(A)** $T(s \to s') = \dfrac{\pi(s')}{C(s)}$ where $C(s) = \sum_{s'' \in \mathcal{N}(s)} \pi(s'')$
- [ ] **(B)** $T(s \to s') = \dfrac{1}{|\mathcal{N}(s)|}$ (uniform over the neighborhood)
- [ ] **(C)** $T(s \to s') = \dfrac{\pi(s)}{C(s')}$
- [ ] **(D)** $T(s \to s') = \exp(-\beta [E(s') + E(s)])$

In [22]:
ws.render_section('q16_16_heat_bath_acceptance_ratio_formula')

**Q16 · medium** · 💡

When $T(s \to s') = \pi(s')/C(s)$ and $T(s' \to s) = \pi(s)/C(s')$ are substituted into the generalized Metropolis–Hastings acceptance formula $A(s \to s') = \min\!\left\{1, \frac{T(s'\to s)\pi(s')}{T(s\to s')\pi(s)}\right\}$, what does $A(s \to s')$ simplify to?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Cancel $\pi(s)$ and $\pi(s')$ between numerator and denominator.

</div>
</details>

- [ ] **(A)** $A(s \to s') = \min\!\left\{1,\; \dfrac{C(s)}{C(s')}\right\}$
- [ ] **(B)** $A(s \to s') = \min\!\left\{1,\; \dfrac{\pi(s')}{\pi(s)}\right\}$
- [ ] **(C)** $A(s \to s') = \min\!\left\{1,\; C(s) C(s')\right\}$
- [ ] **(D)** $A(s \to s') = 0$

In [23]:
ws.render_section('q16_17_heat_bath_unit_acceptance_condition')

**Q17 · easy** · 💡

Under what condition is the Heat-Bath acceptance probability $A(s \to s')$ guaranteed to equal **$1$ ($100\%$ acceptance)** without any rejections?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect $A(s \to s') = \min\{1, C(s)/C(s')\}$.

</div>
</details>

- [ ] **(A)** When the neighborhood normalization sum $C(s)$ is independent of the state $s$ ($C(s) = C(s')$), as when updating a single lattice spin $\sigma_i$ over its full local state set while all other spins remain fixed.
- [ ] **(B)** Only in the infinite-temperature limit $\beta \to 0$.
- [ ] **(C)** Only when the energy of all states in the neighborhood is zero.
- [ ] **(D)** When the neighborhood $\mathcal{N}(s)$ contains no states.

In [24]:
ws.render_section('q16_18_heat_bath_3state_potts_numeric')

**Q18 · medium** · 💡

Consider a 3-state site variable $s_i \in \{1, 2, 3\}$ on a crystal lattice (with all surrounding sites held fixed). In this local environment, the three states have relative Boltzmann weights $\pi(1) = 2.0$, $\pi(2) = 3.0$, and $\pi(3) = 5.0$. Using the single-site **Heat-Bath** algorithm, what is the probability $P(s_i \to 3)$ that site $i$ is placed in state $3$ on the next step?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Normalize the Boltzmann weight of the target state over the sum of weights of all accessible local states.

</div>
</details>

_Numeric answer required._

In [25]:
ws.render_section('q16_19_heat_bath_moving_window_C_ratio')

**Q19 · hard** · 💡

Suppose a particle on a 1D lattice can hop to any site $s'$ within a moving window $\mathcal{N}(s) = \{s-1, s, s+1\}$ centered on its current site $s$. Because the window shifts when the particle moves from $s$ to $s' = s+1$, the local normalization sum changes from $C(s) = \pi(s-1) + \pi(s) + \pi(s+1) = 6.0$ to $C(s') = \pi(s) + \pi(s+1) + \pi(s+2) = 8.0$. What is the Heat-Bath acceptance probability $A(s \to s')$ for this move?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the acceptance probability ratio using the local normalization partition sums for the initial and final neighborhoods.

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q16_20_heat_bath_ising_spin_calc')

**Q20 · medium** · 💡

In a 2D Ising model with energy $E = -J \sum_{\langle ij \rangle} \sigma_i \sigma_j$, a spin $\sigma_i$ experiences a dimensionless local field $\beta h_{\text{loc}} = \beta J \sum_{j \in nn(i)} \sigma_j = +0.50$ (so its local Boltzmann weight is $\propto e^{+0.50\,\sigma_i}$ for $\sigma_i \in \{-1, +1\}$). Under the **Heat-Bath** update rule, what is the probability of setting $\sigma_i' = +1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Normalize the Boltzmann weight for the spin-up state over the two possible orientations $\sigma_i \in \{-1, +1\}$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q16_21_heat_bath_computational_limitation')

**Q21 · easy** · 💡

Why can't we apply exact Heat-Bath sampling directly to a continuous 3D displacement of an atom inside a cube of side $\Delta$ for an arbitrary Lennard-Jones fluid?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the first yellow box at the bottom of.

</div>
</details>

- [ ] **(A)** Because evaluating the continuous normalization integral $C(\mathbf{r}) = \int_{\text{cube}} d\mathbf{r}'\, e^{-\beta V(\mathbf{r}')}$ exactly at every step is computationally intractable for a non-linear potential $V(\mathbf{r}')$.
- [ ] **(B)** Because continuous coordinates do not obey the Boltzmann distribution.
- [ ] **(C)** Because Heat-Bath violates row normalization in 3 dimensions.
- [ ] **(D)** Because $C(\mathbf{r})$ is always negative for Lennard-Jones potentials.

In [28]:
ws.render_section('sec_force_bias_mc')

### Using Local Forces to Push the Walk in the Right Direction
In a dense liquid or solid, blind uniform cube moves frequently push atoms *toward* repulsive neighbors (uphill in energy, $\Delta V \gg k_B T$), resulting in high rejection rates unless $\Delta$ is kept small. How can we approximate the local Heat-Bath distribution $T(\mathbf{r} \to \mathbf{r}') \propto e^{-\beta V(\mathbf{r}')}$ inside a cube so that it *can* be sampled and normalized analytically?

We **Taylor-expand the potential energy $V(\mathbf{r}')$ to first order** around the current position $\mathbf{r}$:
$$V(\mathbf{r}') \approx V(\mathbf{r}) + \nabla V(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r}) = V(\mathbf{r}) - \mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r})$$
where $\mathbf{F}(\mathbf{r}) = -\nabla V(\mathbf{r})$ is the instantaneous mechanical force on the particle!

### The Force-Bias Proposal Distribution Inside a Cube
Introduced by **Pangali, Rao, and Berne (1978)**, **Force-Bias Monte Carlo** restricts trial displacements $\delta\mathbf{r} = \mathbf{r}' - \mathbf{r}$ to a cube $\delta x, \delta y, \delta z \in [-\Delta/2, +\Delta/2]$, and chooses the proposal density:
$$T(\mathbf{r} \to \mathbf{r}') = \frac{1}{C(\mathbf{r})} \exp\!\big[\lambda \beta \mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r})\big] \qquad \text{for } \mathbf{r}' - \mathbf{r} \in \left[-\frac{\Delta}{2}, +\frac{\Delta}{2}\right]^3$$
where $\lambda \in [0, 1]$ (written as $\alpha$ ) is a constant bias parameter (typically $\lambda = 1/2$ or $\lambda = 1$).
- **Why is this easy to sample and normalize?** Because the exponent $\lambda \beta \mathbf{F}\cdot\delta\mathbf{r} = \lambda\beta(F_x \delta x + F_y \delta y + F_z \delta z)$ **factors into three independent 1D exponentials**!
 $$C(\mathbf{r}) = \int_{-\Delta/2}^{+\Delta/2} d(\delta x)\, e^{\lambda\beta F_x \delta x} \int_{-\Delta/2}^{+\Delta/2} d(\delta y)\, e^{\lambda\beta F_y \delta y} \int_{-\Delta/2}^{+\Delta/2} d(\delta z)\, e^{\lambda\beta F_z \delta z} = \prod_{\mu=x,y,z} \frac{2\sinh(\lambda\beta F_\mu \Delta / 2)}{\lambda\beta F_\mu}$$
- **Analytical Sampling via Inverse-CDF Mapping:** Because each 1D exponential $p(\delta x) \propto e^{\lambda\beta F_x \delta x}$ on $[-\Delta/2, +\Delta/2]$ has an elementary cumulative distribution function, we can draw $\delta x, \delta y, \delta z$ directly from three uniform random numbers $u_x, u_y, u_z \in (0, 1)$ using the **mapping (inverse-transform) method** from Lesson 14!

### Force-Bias Acceptance Probability & Why $\lambda = 1/2$ Beats $\lambda = 1$
Substituting $T(\mathbf{r} \to \mathbf{r}')$ and the reverse proposal $T(\mathbf{r}' \to \mathbf{r}) = C(\mathbf{r}')^{-1} \exp[\lambda\beta\mathbf{F}(\mathbf{r}')\cdot(\mathbf{r} - \mathbf{r}')]$ into Metropolis–Hastings gives:
$$A_{\text{FB}}(\mathbf{r} \to \mathbf{r}') = \min\!\left\{1,\; \frac{C(\mathbf{r})}{C(\mathbf{r}')} \exp\!\Big(-\beta\big[V(\mathbf{r}') - V(\mathbf{r}) + \lambda\big(\mathbf{F}(\mathbf{r}) + \mathbf{F}(\mathbf{r}')\big)\cdot(\mathbf{r}' - \mathbf{r})\big]\Big)\right\}$$
Look closely at the exponent: by the trapezoidal rule of calculus, the energy difference along the step is:
$$V(\mathbf{r}') - V(\mathbf{r}) = -\frac{1}{2}\big(\mathbf{F}(\mathbf{r}) + \mathbf{F}(\mathbf{r}')\big)\cdot(\mathbf{r}' - \mathbf{r}) + \mathcal{O}(\Delta^3)$$
Therefore, choosing **$\lambda = 1/2$** makes the force term in the exponent **cancel $V(\mathbf{r}') - V(\mathbf{r})$ to second order in $\Delta$**, yielding a much higher acceptance rate and faster diffusion than $\lambda = 1$ (as shown in Figs. 1 and 3 )!


In [29]:
ws.render_section('q16_22_force_bias_taylor_expansion')

**Q22 · easy** · 💡

In Force-Bias Monte Carlo, how is the local mechanical force $\mathbf{F}(\mathbf{r}) = -\nabla V(\mathbf{r})$ introduced to bias trial moves toward regions of higher Boltzmann probability?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Taylor-expand $V(\mathbf{r}')$ to first order about the current coordinate $\mathbf{r}$.

</div>
</details>

- [ ] **(A)** By Taylor-expanding $V(\mathbf{r}') \approx V(\mathbf{r}) - \mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r})$ inside a trial cube and setting $T(\mathbf{r} \to \mathbf{r}') \propto \exp[\lambda\beta\mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r})]$.
- [ ] **(B)** By integrating $\mathbf{F} = m\mathbf{a}$ for $100$ Verlet steps without any acceptance test.
- [ ] **(C)** By rejecting all moves where $\mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r}) > 0$.
- [ ] **(D)** By setting the temperature $\beta$ proportional to $|\mathbf{F}(\mathbf{r})|$.

In [30]:
ws.render_section('q16_23_force_bias_sampling_method')

**Q23 · easy** · 💡

how do we sample a trial displacement $\delta\mathbf{r} = \mathbf{r}' - \mathbf{r}$ from the exponential Force-Bias distribution $T(\mathbf{r} \to \mathbf{r}') \propto \exp[\lambda\beta\mathbf{F}(\mathbf{r})\cdot(\mathbf{r}' - \mathbf{r})]$ inside a cube $[ -\Delta/2, +\Delta/2 ]^3$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall from Lesson 14 how a 1D exponential PDF on a finite interval is sampled from a uniform random number.

</div>
</details>

- [ ] **(A)** Using the analytical inverse-CDF mapping method along each Cartesian coordinate $x, y, z$ independently.
- [ ] **(B)** Using the Box–Muller polar method for unbounded Gaussians.
- [ ] **(C)** By running a secondary sub-Markov chain of $10^4$ steps inside the cube.
- [ ] **(D)** By always placing $\mathbf{r}'$ at the corner of the cube.

In [31]:
ws.render_section('q16_24_force_bias_1d_normalization_calc')

**Q24 · medium** · 💡

In a 1D Force-Bias move on $\delta x \in [-\Delta/2, +\Delta/2]$ with interval width $\Delta = 1.0\text{ Å}$ and force-bias parameter $\lambda \beta F_x = 2.0\text{ Å}^{-1}$, what is the numerical value of the proposal normalization constant $C(x) = \int_{-\Delta/2}^{+\Delta/2} e^{\lambda\beta F_x\,\delta x}\, d(\delta x)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the exponential force-bias factor over the symmetric displacement interval $[-\Delta/2, +\Delta/2]$.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q16_25_force_bias_forward_reverse_ratio')

**Q25 · medium** · 💡

In a 1D Force-Bias step from $x \to x'$ with displacement $\delta x = x' - x = +0.20\text{ Å}$, suppose $\beta = 1.0$, $\lambda = 0.5$, the initial force is $F(x) = +3.0$, the final force is $F(x') = +1.0$, and the normalization ratio is $C(x)/C(x') = 1.10$. What is the reverse-to-forward proposal ratio:
$$\frac{T(x' \to x)}{T(x \to x')} = \frac{C(x)}{C(x')} \exp\!\big[-\lambda\beta\big(F(x) + F(x')\big)(x' - x)\big]$$

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the product of the normalization ratio and the exponential of the sum of forward and reverse biasing forces.

</div>
</details>

_Numeric answer required._

In [33]:
ws.render_section('q16_26_why_lambda_half_beats_lambda_one')

**Q26 · medium** · 💡

In the liquid Argon benchmarks, why does Force-Bias MC with $\lambda = 1/2$ achieve a significantly higher acceptance probability and faster diffusion than Force-Bias MC with $\lambda = 1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $\Delta V = V(\mathbf{r}') - V(\mathbf{r})$ with $-\lambda(\mathbf{F}(\mathbf{r}) + \mathbf{F}(\mathbf{r}'))\cdot(\mathbf{r}' - \mathbf{r})$.

</div>
</details>

- [ ] **(A)** Because the reverse proposal uses $\mathbf{F}(\mathbf{r}')$ while the forward proposal uses $\mathbf{F}(\mathbf{r})$, their sum in the acceptance ratio forms the trapezoidal force average $\frac{1}{2}[\mathbf{F}(\mathbf{r}) + \mathbf{F}(\mathbf{r}')]\cdot\Delta\mathbf{r}$ when $\lambda = 1/2$, canceling $\Delta V$ to second order!
- [ ] **(B)** Because $\lambda = 1$ violates detailed balance, whereas $\lambda = 1/2$ satisfies it.
- [ ] **(C)** Because $\lambda = 1/2$ does not require computing forces at the new position $\mathbf{r}'$.
- [ ] **(D)** Because $\lambda = 1$ makes the proposal distribution uniform.

In [34]:
ws.render_section('q16_27_force_bias_zero_force_limit')

**Q27 · easy** · 💡

What does the Force-Bias Monte Carlo algorithm reduce to when $\lambda \to 0$ (or when the forces vanish, $\mathbf{F}(\mathbf{r}) = \mathbf{0}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $T(\mathbf{r} \to \mathbf{r}') \propto \exp[\lambda\beta\mathbf{F}\cdot(\mathbf{r}' - \mathbf{r})]$ on the cube $[ -\Delta/2, +\Delta/2 ]^3$ at $\lambda = 0$.

</div>
</details>

- [ ] **(A)** It reduces identically to the classic Metropolis algorithm with a uniform proposal inside the cube of side $\Delta$.
- [ ] **(B)** It reduces to deterministic Newton–Verlet Molecular Dynamics.
- [ ] **(C)** The acceptance probability drops to zero for all moves.
- [ ] **(D)** The trial step size $\Delta$ becomes infinite.

In [35]:
ws.render_section('sec_smart_mc_and_brownian')

### Smart Monte Carlo: Displaced Gaussian Proposals
In Force-Bias MC, restricting moves to a hard cube requires computing the state-dependent normalization constant $C(\mathbf{r}) = \prod_\mu \frac{2\sinh(\lambda\beta F_\mu \Delta/2)}{\lambda\beta F_\mu}$. **Rossky, Doll, and Friedman (1978)** realized that if we replace the hard cube with a **displaced Gaussian probability density** over $(-\infty, +\infty)^3$, the normalization constant becomes **completely independent of the force $\mathbf{F}(\mathbf{r})$**!

In **Smart Monte Carlo**, a particle at $\mathbf{r}$ proposes a trial displacement $\Delta\mathbf{r} = \mathbf{r}' - \mathbf{r}$ composed of a deterministic drift along the force plus isotropic Gaussian white noise $\boldsymbol{\delta}$:
$$\Delta\mathbf{r} = \mathbf{r}' - \mathbf{r} = \tau \beta \mathbf{F}(\mathbf{r}) + \boldsymbol{\delta}, \qquad \langle \boldsymbol{\delta} \rangle = \mathbf{0}, \quad \langle \delta_\mu^2 \rangle = 2\tau$$
This generates the forward Gaussian transition probability density:
$$T(\mathbf{r} \to \mathbf{r}') = \frac{1}{(4\pi\tau)^{3/2}} \exp\!\left[-\frac{\big(\mathbf{r}' - \mathbf{r} - \tau\beta\mathbf{F}(\mathbf{r})\big)^2}{4\tau}\right]$$
and the corresponding reverse Gaussian transition density starting from $\mathbf{r}'$:
$$T(\mathbf{r}' \to \mathbf{r}) = \frac{1}{(4\pi\tau)^{3/2}} \exp\!\left[-\frac{\big(\mathbf{r} - \mathbf{r}' - \tau\beta\mathbf{F}(\mathbf{r}')\big)^2}{4\tau}\right]$$

### Exact Derivation of the Smart MC Acceptance Probability
Let us form the ratio $\frac{T(\mathbf{r}' \to \mathbf{r})\,\pi(\mathbf{r}')}{T(\mathbf{r} \to \mathbf{r}')\,\pi(\mathbf{r})}$. First, the prefactors $(4\pi\tau)^{-3/2}$ cancel out! Expanding the difference of the two Gaussian exponents gives:
$$\ln\!\left[\frac{T(\mathbf{r}' \to \mathbf{r})}{T(\mathbf{r} \to \mathbf{r}')}\right] = \frac{\big(\Delta\mathbf{r} - \tau\beta\mathbf{F}(\mathbf{r})\big)^2 - \big(-\Delta\mathbf{r} - \tau\beta\mathbf{F}(\mathbf{r}')\big)^2}{4\tau}$$
$$= -\frac{1}{2}\beta\big(\mathbf{F}(\mathbf{r}) + \mathbf{F}(\mathbf{r}')\big)\cdot(\mathbf{r}' - \mathbf{r}) - \frac{\tau\beta^2}{4}\Big(|\mathbf{F}(\mathbf{r}')|^2 - |\mathbf{F}(\mathbf{r})|^2\Big)$$
Multiplying by $\pi(\mathbf{r}')/\pi(\mathbf{r}) = e^{-\beta[V(\mathbf{r}') - V(\mathbf{r})]}$ and factoring $\frac{1}{2}(\mathbf{F}(\mathbf{r}') + \mathbf{F}(\mathbf{r}))$ yields the exact Smart MC acceptance formula:
$$A(\mathbf{r} \to \mathbf{r}') = \min\!\left\{1,\; \exp\!\left(-\beta\left[V(\mathbf{r}') - V(\mathbf{r}) + \frac{\mathbf{F}(\mathbf{r}') + \mathbf{F}(\mathbf{r})}{2}\cdot\left(\mathbf{r}' - \mathbf{r} + \frac{\tau\beta}{2}\big(\mathbf{F}(\mathbf{r}') - \mathbf{F}(\mathbf{r})\big)\right)\right]\right)\right\}$$

### Key Mathematical & Physical Properties of Smart MC
1. **Identically $A = 1$ for Any Linear Potential!** Suppose the potential is linear across the region sampled: $V(\mathbf{r}) = V_0 - \mathbf{F}_0\cdot\mathbf{r}$, so the force is constant: $\mathbf{F}(\mathbf{r}') = \mathbf{F}(\mathbf{r}) = \mathbf{F}_0$. Then:
 - $\mathbf{F}(\mathbf{r}') - \mathbf{F}(\mathbf{r}) = \mathbf{0}$,
 - $\frac{\mathbf{F}(\mathbf{r}') + \mathbf{F}(\mathbf{r})}{2}\cdot(\mathbf{r}' - \mathbf{r}) = \mathbf{F}_0\cdot(\mathbf{r}' - \mathbf{r})$, and
 - $V(\mathbf{r}') - V(\mathbf{r}) = -\mathbf{F}_0\cdot(\mathbf{r}' - \mathbf{r})$.
 The bracket inside the exponent sums to $-\mathbf{F}_0\cdot\Delta\mathbf{r} + \mathbf{F}_0\cdot\Delta\mathbf{r} = 0$, giving **$A(\mathbf{r} \to \mathbf{r}') = e^0 = 1$ for any step size $\tau$!**
2. **Rejection Scales Only with Potential Curvature $\mathcal{O}(\tau^{3/2})$:** Because $|\Delta\mathbf{r}| = \mathcal{O}(\tau^{1/2})$, Taylor-expanding $V(\mathbf{r}')$ and $\mathbf{F}(\mathbf{r}')$ around $\mathbf{r}$ shows that both the $\mathcal{O}(\tau^{1/2})$ linear slope and $\mathcal{O}(\tau)$ terms cancel, leaving the log-acceptance error proportional to third derivatives and force gradients $\nabla\mathbf{F} = -\nabla\nabla V$ at $\mathcal{O}(\tau^{3/2})$.
3. **Infinite Sampling Domain:** Because a Gaussian has tails extending to $\pm\infty$, the sampling domain is infinite in principle (making ergodicity immediate!).

### Connection to Brownian Dynamics & Diffusion Quantum Monte Carlo
Consider a large molecule in a viscous solvent. In the high-viscosity (overdamped) limit, its probability density $\rho(R, t)$ evolves according to the **Smoluchowski (Fokker–Planck) master equation**:
$$\frac{\partial \rho(R, t)}{\partial t} = D \nabla^2 \rho(R, t) - \beta D \nabla \cdot \big(\mathbf{F}(R)\rho(R, t)\big)$$
Integrating this stochastic differential equation over a short time step $\tau$ gives **Brownian Dynamics**:
$$R(t + \tau) = R(t) + \tau \beta D \mathbf{F}(R(t)) + \boldsymbol{\eta}(t), \qquad \langle \boldsymbol{\eta}(t) \rangle = \mathbf{0}, \quad \langle \eta_\mu^2(t) \rangle = 2 D \tau$$
- Look at the transition probability density of one Brownian Dynamics step (setting $D = 1$ or absorbing $D$ into $\tau$):
 $$T(R \to R') \propto \exp\!\left[-\frac{\big(R' - R - \beta D \tau \mathbf{F}(R)\big)^2}{4 D \tau}\right]$$
 *(Note: writes $2D\tau$ in the denominator for $\langle\eta^2\rangle = D\tau$ or $4D\tau$ for $\langle\eta^2\rangle = 2D\tau$.)*
- **Hybrid Method:** Uncorrected Brownian Dynamics has a finite time-step error $\mathcal{O}(\tau)$ because $\mathbf{F}(R)$ is held constant across $\tau$. **Smart Monte Carlo is simply Brownian Dynamics with a Metropolis rejection step tacked on to enforce exact detailed balance at any $\tau$!**
- **Diffusion Quantum Monte Carlo (DMC):** Under a Wick rotation to imaginary time ($t \to -i\hbar\tau$), the Schrödinger equation for a many-body wavefunction becomes an identical drift-diffusion equation (where the "quantum force" $\mathbf{F}_Q = 2\nabla\ln\Psi_T$ guides walkers toward regions of high trial wavefunction amplitude), and DMC uses the exact same Smart MC Metropolis acceptance test to eliminate finite time-step bias!


In [36]:
ws.render_section('concept_smart_mc_asymmetry')

### Interactive Explorer: Smart MC Displaced Gaussians & Linear-Potential Exactness

Vary the Smart MC step parameter $\tau$, trial displacement $\Delta x$, and potential curvature $V''(x)$ to visualize the forward proposal Gaussian $T(x \to x')$ (shifted by $+\tau\beta F(x)$) and reverse proposal Gaussian $T(x' \to x)$ (shifted by $+\tau\beta F(x')$). Set curvature $V'' = 0$ to verify that $A_{\mathrm{Smart}} = 1.0000$ identically on a linear slope!

**Key Takeaway:** In Smart MC, forward and reverse Gaussian proposals are shifted by $+\tau\beta F(x)$ and $+\tau\beta F(x')$. When the potential is locally linear ($V'' = 0 \implies F(x') = F(x)$), the ratio $T(x'\to x)/T(x\to x') = e^{+\beta F_0 \Delta x}$ cancels the Boltzmann factor $e^{-\beta \Delta V} = e^{-\beta F_0 \Delta x}$ **identically**, giving $100\%$ acceptance ($A = 1$) for any $\tau$!

In [37]:
ws.render_section('concept_smart_vs_metropolis_efficiency')

### Interactive Benchmark: Metropolis vs. Force-Bias & Smart MC in Liquid Argon

Switch between the acceptance probability curve (Fig. 1 of ) and the 100-step mean-squared displacement $\langle \Delta r^2(100) \rangle$ (Fig. 3 of ) across step sizes $\Delta/\sigma$ to examine when Force-Bias ($\lambda = 0.5, 1.0$) and Smart MC outperform classic Metropolis—and why classic Metropolis remains competitive!

**Key Takeaway:** Force-Bias ($\lambda=0.5$) and Smart MC achieve substantially higher acceptance ratios than classic Metropolis at a fixed step size $\Delta/\sigma$ and reduce energy autocorrelation $C(50)$ near $\Delta/\sigma \approx 0.20$. However, if you increase the classic Metropolis step size to $\Delta/\sigma \approx 0.34$ ($A \approx 0.20$), classic Metropolis achieves comparable or higher raw diffusion without the extra CPU cost of computing forces!

In [38]:
ws.render_section('q16_28_smart_mc_trial_displacement_mean_variance')

**Q28 · easy** · 💡

In Smart Monte Carlo, the trial displacement of a particle is generated as $\Delta\mathbf{r} = \mathbf{r}' - \mathbf{r} = \tau\beta\mathbf{F}(\mathbf{r}) + \boldsymbol{\delta}$, where each Cartesian component of $\boldsymbol{\delta}$ is drawn from a zero-mean Gaussian with $\langle \delta_\mu^2 \rangle = 2\tau$. What are the mean $\langle \Delta x \rangle$ and variance $\operatorname{Var}(\Delta x) = \langle (\Delta x - \langle \Delta x \rangle)^2 \rangle$ of the proposed $x$-displacement?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Add the deterministic drift $\tau\beta F_x(\mathbf{r})$ to the zero-mean noise $\delta_x$.

</div>
</details>

- [ ] **(A)** Mean $\langle \Delta x \rangle = \tau\beta F_x(\mathbf{r})$ and variance $\operatorname{Var}(\Delta x) = 2\tau$.
- [ ] **(B)** Mean $\langle \Delta x \rangle = 0$ and variance $\operatorname{Var}(\Delta x) = \tau\beta F_x(\mathbf{r})$.
- [ ] **(C)** Mean $\langle \Delta x \rangle = 2\tau$ and variance $\operatorname{Var}(\Delta x) = 4\pi\tau$.
- [ ] **(D)** Mean $\langle \Delta x \rangle = -F_x(\mathbf{r})$ and variance $\operatorname{Var}(\Delta x) = 0$.

In [39]:
ws.render_section('q16_29_smart_mc_gaussian_normalization_advantage')

**Q29 · medium** · 💡

What is the key mathematical simplification of **Smart Monte Carlo's** unbounded displaced-Gaussian proposal $T(\mathbf{r} \to \mathbf{r}') = (4\pi\tau)^{-3/2}\exp\!\left[-\frac{(\mathbf{r}' - \mathbf{r} - \tau\beta\mathbf{F}(\mathbf{r}))^2}{4\tau}\right]$ compared to **Force-Bias Monte Carlo's** truncated cube proposal?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the normalization factor $(4\pi\tau)^{-3/2}$ of a shifted Gaussian over $(-\infty, \infty)^3$ with the cube normalization $C(\mathbf{r}) = \prod_\mu \frac{2\sinh(\lambda\beta F_\mu \Delta/2)}{\lambda\beta F_\mu}$.

</div>
</details>

- [ ] **(A)** Because shifting the mean of a Gaussian over $(-\infty, +\infty)^3$ does not change its integral, the normalization factor $(4\pi\tau)^{-3/2}$ is completely independent of $\mathbf{F}(\mathbf{r})$ and cancels out of $T(\mathbf{r}' \to \mathbf{r}) / T(\mathbf{r} \to \mathbf{r}')$!
- [ ] **(B)** Smart MC does not require evaluating the force $\mathbf{F}(\mathbf{r}')$ at the trial position $\mathbf{r}'$.
- [ ] **(C)** Smart MC makes $T(\mathbf{r} \to \mathbf{r}') = T(\mathbf{r}' \to \mathbf{r})$ symmetric even when $\mathbf{F}(\mathbf{r}) \ne \mathbf{0}$.
- [ ] **(D)** Smart MC restricts displacements to a sphere of radius $\tau$.

In [40]:
ws.render_section('q16_30_smart_mc_linear_potential_exact_acceptance')

**Q30 · easy** · 💡

A particle moves in a 3D linear potential $V(\mathbf{r}) = V_0 - \mathbf{F}_0 \cdot \mathbf{r}$ with a constant force $\mathbf{F}(\mathbf{r}) = \mathbf{F}_0 \ne \mathbf{0}$ everywhere. In a Smart Monte Carlo simulation with step parameter $\tau = 0.50\text{ Å}^2$ at $\beta = 2.0\text{ eV}^{-1}$, a large trial move $\Delta\mathbf{r} = \mathbf{r}' - \mathbf{r}$ increases the potential energy by $\Delta V = +3.0\text{ eV}$. What is the exact Smart MC acceptance probability $A(\mathbf{r} \to \mathbf{r}')$ for this move?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $\mathbf{F}(\mathbf{r}') = \mathbf{F}(\mathbf{r}) = \mathbf{F}_0$ and $\Delta V = -\mathbf{F}_0\cdot(\mathbf{r}' - \mathbf{r})$ into the Smart MC acceptance formula.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q16_31_smart_mc_harmonic_acceptance_numeric')

**Q31 · hard** · 💡

Consider a 1D particle at $x = 0$ in a harmonic potential $V(x) = \frac{1}{2}k x^2$ with spring constant $k = 2.0$ (so $V(x) = x^2$ and force $F(x) = -V'(x) = -2x$) at $\beta = 1.0$ and Smart MC parameter $\tau = 0.25$. Suppose the particle proposes a step from $x = 0$ to $x' = 1.0$:
- $V(x) = 0$, $V(x') = 1.0 \implies \Delta V = +1.0$
- $F(x) = 0$, $F(x') = -2.0$
- $\Delta x = x' - x = +1.0$

Using the Smart MC acceptance formula:
$$A(x \to x') = \min\!\left\{1,\; \exp\!\left(-\beta\left[\Delta V + \frac{F(x') + F(x)}{2}\left(\Delta x + \frac{\tau\beta}{2}\big(F(x') - F(x)\big)\right)\right]\right)\right\}$$
what is the Smart MC acceptance probability $A(0 \to 1.0)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the terms inside the exponent of the Smart MC acceptance formula using the initial and final forces and the potential energy difference.

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q16_32_smart_mc_rejection_curvature_order')

**Q32 · hard** · 💡

For a smooth anharmonic potential $V(\mathbf{r})$ where typical Smart MC trial displacements scale as $|\Delta\mathbf{r}| = \mathcal{O}(\tau^{1/2})$, what physical derivative of $V(\mathbf{r})$ causes the Smart MC acceptance probability to deviate from $1$, and at what order in $\tau$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $A = 1$ for any linear potential ($\nabla\nabla V = 0$), check the lowest non-vanishing term when $V(\mathbf{r}')$ and $\mathbf{F}(\mathbf{r}')$ are Taylor-expanded around $\mathbf{r}$.

</div>
</details>

- [ ] **(A)** Potential curvature (the Hessian $\nabla\nabla V = -\nabla\mathbf{F}$) and higher derivatives, entering the log-acceptance exponent at order $\mathcal{O}(\tau^{3/2})$ (from the $\frac{\tau\beta^2}{4}(|\mathbf{F}'|^2 - |\mathbf{F}|^2)$ and cubic $\mathcal{O}(\Delta r^3)$ terms).
- [ ] **(B)** The linear force $-\nabla V$, entering at order $\mathcal{O}(\tau^{1/2})$ just like classic Metropolis.
- [ ] **(C)** The constant reference energy $V(\mathbf{r}_0)$, entering at order $\mathcal{O}(1)$.
- [ ] **(D)** The particle mass $m$, entering at order $\mathcal{O}(\tau^2)$.

In [43]:
ws.render_section('q16_33_smoluchowski_equation_terms')

**Q33 · medium** · 💡

the high-viscosity (overdamped) master equation for a molecule in a solvent is the **Smoluchowski equation**:
$$\frac{\partial \rho(R, t)}{\partial t} = D \nabla^2 \rho(R, t) - \beta D \nabla \cdot \big(\mathbf{F}(R) \rho(R, t)\big)$$
What are the physical meanings of the two terms on the right-hand side, and why does the same coefficient $D$ appear in both?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Connect $D\nabla^2\rho$ to Fickian thermal diffusion and $-\beta D\nabla\cdot(\mathbf{F}\rho)$ to force-driven drift via the Einstein fluctuation-dissipation relation.

</div>
</details>

- [ ] **(A)** $D\nabla^2\rho$ is isotropic thermal diffusion driven by random solvent collisions, $-\beta D\nabla\cdot(\mathbf{F}\rho)$ is deterministic drift along the mechanical force $\mathbf{F}$, and their shared coefficient $D$ (Einstein relation $\mu = \beta D$) ensures that $\rho_{\text{eq}}(R) \propto e^{-\beta V(R)}$ is the stationary solution ($\partial\rho/\partial t = 0$).
- [ ] **(B)** Both terms represent inertial kinetic energy conservation in a vacuum.
- [ ] **(C)** $D\nabla^2\rho$ is the quantum kinetic energy and $-\beta D\nabla\cdot(\mathbf{F}\rho)$ is the Pauli exclusion principle.
- [ ] **(D)** The two terms cancel only when $T = 0\text{ K}$.

In [44]:
ws.render_section('q16_34_brownian_dynamics_to_smart_mc')

**Q34 · medium** · 💡

How does **Smart Monte Carlo** relate to a discrete time step $R(t + \tau) = R(t) + \tau\beta D\mathbf{F}(R(t)) + \boldsymbol{\eta}(t)$ of **Brownian Dynamics**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens to the finite-time-step error of Euler–Maruyama Brownian Dynamics when a Metropolis rejection step is added.

</div>
</details>

- [ ] **(A)** One step of Brownian Dynamics is identical to the Smart MC trial proposal $T(R \to R')$; adding the Smart MC Metropolis rejection test eliminates the finite-$\tau$ discretization error of Brownian Dynamics and enforces exact detailed balance!
- [ ] **(B)** Brownian Dynamics is already exact for any finite time step $\tau$ without rejections.
- [ ] **(C)** Smart MC is the zero-viscosity ($D \to \infty$) limit of Newtonian Molecular Dynamics.
- [ ] **(D)** Brownian Dynamics moves particles opposite to the force $\mathbf{F}(R)$.

In [45]:
ws.render_section('q16_35_diffusion_qmc_connection')

**Q35 · medium** · 💡

As noted at the bottom of and 14, which quantum simulation method uses the exact same drift-diffusion equation and Smart MC Metropolis acceptance step to sample many-body ground states?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the orange box at the bottom of.

</div>
</details>

- [ ] **(A)** Diffusion Quantum Monte Carlo (DMC), where imaginary-time Schrödinger evolution combines diffusion with drift along the quantum force $\mathbf{F}_Q = 2\nabla\ln\Psi_T$.
- [ ] **(B)** Hartree–Fock self-consistent field theory.
- [ ] **(C)** Density Functional Perturbation Theory (DFPT).
- [ ] **(D)** Tight-binding molecular dynamics.

In [46]:
ws.render_section('q16_36_cube_vs_gaussian_variance_matching')

**Q36 · medium** · 💡

To fairly compare a 1D uniform cube move $\delta x \in [-\Delta/2, +\Delta/2]$ against a 1D Smart MC Gaussian noise step $\delta_x$ with variance $2\tau$, we match their bare step variances. If $\tau / \sigma^2 = 0.0024$, what is the equivalent dimensionless cube width $\Delta / \sigma$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate the variance of a uniform distribution of width $\Delta$ to the Gaussian noise variance $2\tau$ and solve for $\Delta$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q16_37_does_force_bias_always_win_cpu')

**Q37 · medium** · 💡

Look at the question posed: *'Force bias has higher acceptance ratio than classic Metropolis. Does it lead to faster convergence?'* Based on Figs. 1–4, what is the nuanced practical conclusion for simple Lennard-Jones liquids like Argon?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the peak height of Fig. 3 ($\langle \Delta r^2(100) \rangle$) and the plateau of Fig. 4 ($\sigma_V$), keeping in mind that Force-Bias/Smart MC must also compute $\mathbf{F}(\mathbf{r}')$ at every step.

</div>
</details>

- [ ] **(A)** While Force-Bias ($\lambda=0.5$) and Smart MC have higher acceptance ratios at a given $\Delta$ and lower energy correlation $C(50)$ at moderate $\Delta$, classic Metropolis at a larger step size ($\Delta/\sigma \approx 0.36$, $A \approx 20\%$) achieves comparable or higher 100-step diffusion and similar energy variance without the extra CPU cost of computing forces!
- [ ] **(B)** Force-Bias MC is $1000\times$ faster per CPU second than classic Metropolis for every observable at every step size.
- [ ] **(C)** Force-Bias and Smart MC always have a lower acceptance ratio than classic Metropolis.
- [ ] **(D)** Classic Metropolis cannot converge for liquid Argon at any step size.

In [48]:
ws.render_section('q16_38_summary_smarter_mc_methods')

**Q38 · medium** · 💡

Select **all** true statements summarizing the five 'Smarter Monte Carlo' techniques covered in Lesson 16:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Verify each statement against the derivations in Sections 1–5.

</div>
</details>

- [ ] **(A)** A deterministic cycle of moves $(A, B, A, B, \dots)$ violates detailed balance in general ($P_A P_B \ne P_B P_A$), yet still converges to the exact equilibrium distribution $\pi$ via global balance $\pi P_A P_B = \pi$.
- [ ] **(B)** Preferential sampling selects particle $i$ with probability $h(z_i)/H(\mathbf{R})$ and requires the asymmetric factor $[h(z_i')/H(\mathbf{R}')]/[h(z_i)/H(\mathbf{R})]$ in the Metropolis test.
- [ ] **(C)** Heat-Bath MC samples a local neighborhood from $T(s\to s') = \pi(s')/C(s)$ and achieves $100\%$ acceptance ($A = 1$) whenever the local normalization $C(s) = C(s')$ is state-independent.
- [ ] **(D)** Smart MC proposes a force-drifted Gaussian step $\Delta\mathbf{r} = \tau\beta\mathbf{F}(\mathbf{r}) + \boldsymbol{\delta}$ that achieves $A = 1$ identically for any linear potential and provides a time-step-error-free hybrid version of Brownian Dynamics.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()